# 3.2 Building Basic Agents with LangChain & AutoGen

**Week 4 — Agentic AI & Multi-Agent Systems**

## Learning objectives
- Build a LangChain-style agent using prompt templates and a custom tool
- Understand tool selection logic, fallback strategies, and hallucination risk
- Understand AutoGen's two foundational building blocks: `AssistantAgent` and `UserProxyAgent`
- Stream an agent's reasoning to the console in real time
- Understand how multimodal agents extend the same loop to image inputs

> This notebook is written so it runs **fully offline**. Where a real LangChain / AutoGen /
> OpenAI / Anthropic call would normally go, you'll see a clearly marked mock plus the real code
> as a comment. Install the real packages (`pip install langchain langchain-anthropic pyautogen`)
> and swap in an API key to run the production version.


## 1. LangChain Agent Foundations

LangChain structures an agent around three pieces:

1. **Prompt templates** — reusable, parameterised prompts (so you don't hand-write a new prompt
   string every time).
2. **Tools** — plain Python functions wrapped with a name + description so the LLM can choose them.
3. **An agent executor** — the loop that repeatedly calls the LLM, parses its tool choice, executes
   the tool, and feeds the result back in (this *is* the ReAct loop from 3.1, implemented for you).


In [ ]:
from string import Template

# --- 1. Prompt template ---
AGENT_PROMPT = Template("""You are a helpful assistant with access to these tools:
$tool_descriptions

Goal: $goal
History so far: $history

Decide the single next step. Respond with either:
  ACTION: <tool_name> | <tool_input>
  or
  FINAL: <answer>
""")

def render_prompt(goal, tools, history):
    tool_desc = "\n".join(f"- {name}: {t.description}" for name, t in tools.items())
    hist_text = "\n".join(history) if history else "(none yet)"
    return AGENT_PROMPT.substitute(tool_descriptions=tool_desc, goal=goal, history=hist_text)

print(render_prompt("Convert 100 USD to INR", {}, []))


In [ ]:
# --- 2. Tools: converting a plain Python function into a LangChain-style tool ---
from dataclasses import dataclass
from typing import Callable, Dict

@dataclass
class Tool:
    name: str
    description: str
    func: Callable[[str], str]

def fx_convert(query: str) -> str:
    """query format: 'AMOUNT FROM_CCY to TO_CCY', e.g. '100 USD to INR'"""
    rates = {("USD", "INR"): 83.1, ("EUR", "INR"): 90.4, ("GBP", "INR"): 105.2}
    try:
        amount, from_ccy, _, to_ccy = query.split()
        rate = rates.get((from_ccy.upper(), to_ccy.upper()))
        if rate is None:
            return f"No rate available for {from_ccy}->{to_ccy}"
        return f"{amount} {from_ccy.upper()} = {float(amount) * rate:.2f} {to_ccy.upper()}"
    except Exception:
        return "Could not parse query. Expected format: '100 USD to INR'"

def web_search_stub(query: str) -> str:
    """Stand-in for LangChain's DuckDuckGoSearchRun tool."""
    fake_index = {"exchange rate policy india": "RBI manages a managed-float exchange rate regime."}
    return fake_index.get(query.lower(), "No results found (this is a stub search tool).")

tools = {
    "fx_convert": Tool("fx_convert", "Convert an amount between currencies. Input: '100 USD to INR'", fx_convert),
    "web_search": Tool("web_search", "Search the web for general knowledge questions.", web_search_stub),
}
print("Tools ready:", list(tools.keys()))


### Tool selection logic, fallback strategies, and hallucination risk

A naive agent will sometimes:
- pick the **wrong tool** for the job,
- pick **no tool** and hallucinate an answer instead of looking something up,
- get a **malformed tool response** and not know how to recover.

Good agent design adds explicit fallback handling around each of these failure modes.


In [ ]:
import re

def mock_llm_router(goal: str, tools: Dict[str, Tool]) -> Dict[str, str]:
    """Stands in for an LLM tool-selection call. A real implementation sends `render_prompt(...)`
    to the model and parses its ACTION/FINAL response."""
    if re.search(r"\d+\s*[A-Za-z]{3}\s+to\s+[A-Za-z]{3}", goal):
        return {"action": "fx_convert", "input": re.search(r"\d+\s*[A-Za-z]{3}\s+to\s+[A-Za-z]{3}", goal).group()}
    if "policy" in goal.lower() or "regime" in goal.lower():
        return {"action": "web_search", "input": goal.lower()}
    return {"action": None, "input": None}

def run_agent(goal: str, tools: Dict[str, Tool]):
    decision = mock_llm_router(goal, tools)
    if decision["action"] is None:
        # Fallback strategy: no confident tool match -> say so rather than hallucinate an answer
        return "I don't have a tool for this and won't guess — please rephrase or provide more detail."
    tool = tools.get(decision["action"])
    if tool is None:
        return f"Fallback: model chose an unknown tool '{decision['action']}'. Ignoring and asking for clarification."
    result = tool.func(decision["input"])
    return f"[used tool: {tool.name}] {result}"

for g in ["Convert 100 USD to INR", "What is India's exchange rate policy?", "Tell me a joke about clouds"]:
    print(g, "->", run_agent(g, tools))


Notice the **third example** ("Tell me a joke about clouds") — a poorly designed agent might
force this into a tool call it doesn't need, or worse, silently hallucinate a "fact" using a tool whose
result doesn't actually answer the question. Our fallback branch instead admits it has no matching tool.
This is the single most important habit to build early: **an agent that says "I don't know" is safer
than one that confidently guesses.**


## 2. AutoGen Building Blocks: `AssistantAgent` and `UserProxyAgent`

AutoGen models a conversation between two (or more) agents instead of a single agent with tools:

- **`AssistantAgent`** — the *reasoning* agent. It only "thinks" in text/code; it does not execute
  anything itself.
- **`UserProxyAgent`** — the *executor* / human-proxy agent. It can execute code the assistant
  proposes, call tools, and optionally pause for real human input.

This split maps directly onto the Brain + Hands analogy from 3.1, but modelled as **two separate
agents talking to each other** rather than one agent with an internal tool-calling loop.

```python
# Real AutoGen usage (requires `pip install pyautogen` and an API key):
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent(name="assistant", llm_config={"model": "gpt-4o-mini"})
user_proxy = UserProxyAgent(name="user_proxy", human_input_mode="NEVER",
                             code_execution_config={"work_dir": "coding"})

user_proxy.initiate_chat(assistant, message="Write and run Python to compute the 20th Fibonacci number.")
```


In [ ]:
# A minimal offline simulation of the AssistantAgent <-> UserProxyAgent exchange

class MockAssistantAgent:
    """Reasoning only — proposes code, never executes it."""
    def respond(self, message: str) -> str:
        if "fibonacci" in message.lower():
            return (
                "Here is Python to compute it:\n"
                "```python\n"
                "def fib(n):\n"
                "    a, b = 0, 1\n"
                "    for _ in range(n):\n"
                "        a, b = b, a + b\n"
                "    return a\n"
                "print(fib(20))\n"
                "```"
            )
        return "I don't have a plan for that request."

class MockUserProxyAgent:
    """Executor — extracts and runs code blocks the assistant proposes."""
    def execute(self, assistant_message: str) -> str:
        import re, io, contextlib
        match = re.search(r"```python\n(.*?)```", assistant_message, re.S)
        if not match:
            return "(no executable code found)"
        code_str = match.group(1)
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            exec(code_str, {})
        return buf.getvalue().strip()

assistant = MockAssistantAgent()
user_proxy = MockUserProxyAgent()

proposal = assistant.respond("Write and run Python to compute the 20th Fibonacci number.")
print("ASSISTANT PROPOSES:\n", proposal, "\n")
result = user_proxy.execute(proposal)
print("USER_PROXY EXECUTES AND RETURNS:\n", result)


Notice the clean separation of responsibility: `MockAssistantAgent` never calls `exec()`, and
`MockUserProxyAgent` never decides what code to write. This separation is exactly why AutoGen is a good
fit for tasks that involve running untrusted or exploratory code — you can insert a human approval gate
between "propose" and "execute" (see `human_input_mode` in the real API) without touching the reasoning
agent at all.


## 3. Streaming Real-Time Console Responses

Agent tasks can take many seconds, especially across multiple tool calls. Streaming the model's output
token-by-token (or step-by-step) gives users feedback that the system hasn't stalled.

```python
# Real streaming with the Anthropic API
with client.messages.stream(
    model="claude-sonnet-5",
    max_tokens=500,
    messages=[{"role": "user", "content": "Explain agentic AI in 3 sentences."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
```

Below is an offline simulation of the same idea using our ReAct-style loop from 3.1, printing each
step as it becomes available instead of only at the end.


In [ ]:
import time

def stream_agent_steps(steps):
    for step in steps:
        print(step, end="", flush=True)
        time.sleep(0.05)  # simulate token-by-token latency
    print()

stream_agent_steps([
    "Thought: the user wants a currency conversion... ",
    "Action: fx_convert(100 USD to INR)... ",
    "Observation: 100 USD = 8310.00 INR... ",
    "Final: 100 USD converts to approximately ₹8,310."
])


## 4. Multimodal Capabilities — Agents That Can See

Modern LLM APIs accept images alongside text in the same message. An agent can therefore be given a
screenshot, a photo, or a scanned document and reason about it in the same ReAct loop — e.g. "look at
this dashboard screenshot and tell me which metric is red, then look up why."

```python
# Real multimodal call (Anthropic API) — base64-encoded image + text in one message
response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_b64}},
            {"type": "text", "text": "What does this chart show, and is anything trending downward?"}
        ]
    }]
)
```

The important architectural point: **the tool-calling loop doesn't change.** An image is just another
kind of input the Brain can reason over — the Hands (tools) and the loop control-flow (3.1) are
identical to the text-only case.


## Key Takeaways

- LangChain agents = prompt template + tools + an executor loop (a concrete ReAct implementation).
- Good agents need explicit **fallback logic** for "no matching tool" — refusing to guess beats
  hallucinating.
- AutoGen splits reasoning (`AssistantAgent`) from execution (`UserProxyAgent`) — useful whenever code
  or actions need a safety/approval boundary.
- Streaming improves perceived responsiveness for multi-step agent tasks.
- Multimodal input extends what the Brain can reason about without changing the agent's control flow.

## Check your understanding
1. Why is it dangerous for an agent to always force a tool call, even when no tool actually fits?
2. What responsibility does `UserProxyAgent` have that `AssistantAgent` deliberately does not?
3. Why might you want a human-approval step between an AutoGen assistant's code proposal and its
   execution in a production system?

Next: **3.3 Introduction to Multi-Agent Systems** — why a single agent with too many tools becomes a
bottleneck, and how to think in terms of specialised agent "teams" instead.
